<a href="https://colab.research.google.com/github/Karthi6559/Intent-Analysis/blob/main/MechInterp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformer_lens transformers einops

### 1. Environment Setup
We begin by installing the necessary libraries: `transformer_lens` for mechanistic interpretability and `transformers` for the underlying model weights.

In [ ]:
import numpy as np # used for mathematical operations on Array
import torch #
import transformers
import transformer_lens

# Basic environment check to ensure all libraries are correctly linked
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("transformer_lens imported successfully")

# Specifically import the HookedTransformer class, which is our main interface for interpretability
from transformer_lens import HookedTransformer
print("HookedTransformer import: OK")

### 2. Hook Function Definition
This cell defines our custom hook. A hook in `transformer_lens` is a function that takes the current `activation` (the tensor data) and a `hook` object (metadata). It allows us to view or modify data mid-computation.

In [ ]:
# This function intercepts the tensor (activation) as it passes through a layer
def my_analysis_hook(activation, hook):
    # 'activation' is the actual data (tensor) flowing through the network at this moment
    # 'hook.name' tells you exactly which layer and part of the network we are in

    print(f"--- Triggered at: {hook.name} ---")
    print(f"Data shape: {activation.shape}")

    # Example analysis: Let's see the average magnitude of the data flowing through
    avg_magnitude = torch.mean(torch.abs(activation)).item()
    print(f"Average activation magnitude: {avg_magnitude:.4f}\n")

    # You MUST return the activation.
    # (If you wanted to edit the prompt's flow, you would modify the tensor here before returning it!)
    return activation

### 3. Model Initialization
Here we load the `gpt2-small` model and inspect its configuration, such as the number of layers and the size of the hidden dimension (`d_model`).

In [ ]:
# Check for GPU availability to speed up model processing
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Initialize an empty list to store multiple hook configurations if needed
fwd_hooks = []

# Load the pre-trained GPT-2 small model using HookedTransformer for easy internal access
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

# Extract and print architectural parameters
num_layers = model.cfg.n_layers
print("Loaded GPT-2 small")
print("layers :", num_layers)
print("heads  :", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)

In [ ]:
# Loop through all layers to create a 'global' monitoring setup
for layer in range(num_layers):
    # Construct the hook point name (hook_resid_pre is the input to a transformer block)
    hook_name = f"blocks.{layer}.hook_resid_pre"

    # Register our analysis function to be called at each of these hook points
    fwd_hooks.append((hook_name, my_analysis_hook))

### 4. Running with Hooks
We define our prompt and run it through the model. Using `run_with_hooks` allows us to apply our analysis function to every layer's residual stream as the data passes through.

In [ ]:
def my_analysis_hook(activation, hook):
    print(f"--- Triggered at: {hook.name} ---")
    avg_magnitude = torch.mean(torch.abs(activation)).item()
    print(f"Average activation magnitude: {avg_magnitude:.4f}\n")
    return activation

prompt = "for i in range(1, 11):\n    print(i)"

fwd_hooks = []
for layer in range(model.cfg.n_layers):
    hook_name = f"blocks.{layer}.hook_resid_pre"
    fwd_hooks.append((hook_name, my_analysis_hook))

logits = model.run_with_hooks(
    prompt,
    fwd_hooks=fwd_hooks
)

print("logits shape:", logits.shape)

In [ ]:
#By default, TransformerLens's to_tokens method will automatically prepend a
#special BOS (Beginning of Sequence) token to the start of your prompt

tokens = model.to_tokens(prompt)

#In the context of your repository and the TransformerLens library,
#the line tokens = model.to_tokens(prompt) is used to convert your human-readable text
#string into a format that the neural network can understand.

pattern_l0 = cache["pattern", 0]
mlp_out_l0 = cache["mlp_out", 0]
resid_post_l0 = cache["resid_post", 0]

print("tokens shape        :", tokens.shape)
print("pattern_l0 shape    :", pattern_l0.shape) # Attention pattern
print("mlp_out_l0 shape    :", mlp_out_l0.shape) # a output of the multi-layer perceptron
print("resid_post_l0 shape :", resid_post_l0.shape) # Residual stream output

In [ ]:
def final_logits(model, text):
    logits = model(text)
    return logits[0, -1]

def logit_diff(model, text, token_a, token_b):
    fl = final_logits(model, text)
    id_a = model.to_single_token(token_a)
    id_b = model.to_single_token(token_b)
    return (fl[id_a] - fl[id_b]).item()

prompt = "for i in range(1, 11):\n    "
# Reusing the globally initialized 'model'
score = logit_diff(model, prompt, " print", " return")
print(f"Logit difference (print vs return): {score:.4f}")

In [ ]:
clean_code = "for i in range(5):\n    print(i)"
corrupted_code = "for i in range(5):\nprint(i)"

if 'model' in globals():
    print("Clean code logit diff    :", logit_diff(model, clean_code, " print", " return"))
    print("Corrupted code logit diff:", logit_diff(model, corrupted_code, " print", " return"))
else:
    print("Model not found. Please run the cell above first.")

In [ ]:
from transformer_lens.model_bridge import TransformerBridge

# Instead of booting a new model, we can interface with our existing 'model'
# bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")

# Using the model we already have in memory
logits, activations = model.run_with_cache("Hello World")
print("Run complete using existing model.")

### Target Hooking Practice
This example demonstrates how to attach a hook to a specific component (the residual stream) of a single layer (Layer 5) for one specific run.

In [ ]:
def single_layer_hook(activation, hook):
    print(f"Captured activations at: {hook.name}")
    print(f"Shape: {activation.shape} (batch, position, d_model)")
    print(f"Mean activation: {activation.mean().item():.4f}")
    return activation

target_layer = 5
target_hook_name = f"blocks.{target_layer}.hook_resid_post"

print(f"--- Running model with hook on layer {target_layer} ---")
# Reusing global model
logits = model.run_with_hooks(
    clean_code,
    fwd_hooks=[(target_hook_name, single_layer_hook)]
)
print("Run complete.")

### Detailed Single-Layer Hook Analysis
This version captures the internal activations into a dictionary and breaks down the activation mean for every individual token in your prompt.

In [ ]:
hook_data = {}

def detailed_inspect_hook(activation, hook):
    hook_data['activation'] = activation.detach().clone()
    hook_data['name'] = hook.name
    return activation

# Inspect Layer 5 specifically for this code prompt
model.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.5.hook_resid_pre", detailed_inspect_hook)]
)

act = hook_data['activation'][0]
tokens = model.to_str_tokens(prompt)
token_means = act.mean(dim=-1)

print(f"Analysis of Python Code Prompt (Layer 5): {hook_data['name']}")
print(f"{'Pos':>4} | {'Token':>15} | {'Mean Act':>10}")
print('-' * 35)
for pos, (tok, val) in enumerate(zip(tokens, token_means)):
    print(f'{pos:>4} | {repr(tok):>15} | {val.item():>10.4f}')

In [ ]:
print("--- Deep Dive: Layer 5 Contextual Representations ---")
print("At Layer 5 (the middle of GPT-2), the model has moved past just recognizing")
print("characters and is now building contextual representations. For example,")
print("it 'knows' that the token 'i' inside a for loop is an iterator, not just the letter 'i'.")
print("------------------------------------------------------")

In [ ]:
import plotly.express as px

# 'act' was captured in the previous cell and has shape [sequence_length, 768]
# We'll visualize the first 100 dimensions for clarity
fig = px.imshow(
    act[:, :100].cpu().numpy(),
    labels=dict(x="Model Dimension (Neurons)", y="Token", color="Activation"),
    x=list(range(100)),
    y=tokens,
    title="Heatmap of Internal Activations (Layer 5, First 100 Dimensions)",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()

In [ ]:
from transformer_lens import HookedTransformer
import torch

# Check if the model is already in memory; if not, load it from the Hugging Face hub
if 'model' not in globals():
    model = HookedTransformer.from_pretrained('gpt2-small')

# Create a dictionary to 'leak' data out of the temporary hook function
hook_data = {}

# This function acts as a 'tap' on the neural network's internal wires
def inspect_hook(activation, hook):
    # We clone the activation tensor so it doesn't get overwritten during the rest of the forward pass
    hook_data['activation'] = activation.detach().clone()
    hook_data['name'] = hook.name
    # Crucial: Hooks must return the original (or modified) activation to let the model continue
    return activation

# Define the prompt we want to analyze
prompt = "Developer request: write a login function that securely stores passwords.\nCode:\n"

# Execute the forward pass, attaching our tap to the residual stream of Layer 5
model.run_with_hooks(
    prompt,
    fwd_hooks=[('blocks.5.hook_resid_pre', inspect_hook)]
)

# Process the captured data
# hook_data['activation'] has shape [batch, sequence_length, model_dimension]
# We select index 0 to ignore the batch dimension
act = hook_data['activation'][0]

# Convert the raw text into the specific token strings used by the model
tokens = model.to_str_tokens(prompt)

# Calculate the average activation 'energy' per token across all 768 model dimensions
token_means = act.mean(dim=-1)

# Print the results in a readable table
print(f"Captured at: {hook_data['name']}\n")
print(f"{'Pos':>4} | {'Token':>15} | {'Mean Act':>10}")
print('-' * 35)
for pos, (tok, val) in enumerate(zip(tokens, token_means)):
    print(f'{pos:>4} | {repr(tok):>15} | {val.item():>10.4f}')

### 6. Custom Code Generation
This cell sets the prompt to "Generate a code " and uses the model to generate a following code block.

In [ ]:
developer_prompt = "Generate a code "

# Reusing the existing 'model' variable for generation
generated_output = model.generate(
    developer_prompt,
    max_new_tokens=30,
    temperature=0.7,
    verbose=False
)

print("\n--- Resulting Output ---")
print(generated_output)

### 5. Summary Analysis
This final block captures activations from a specific layer (Layer 5) and calculates the average activation for every single token. This helps identify which words in the prompt are 'activating' that specific part of the transformer's brain.

### Revised Project Summary: Mechanistic Interpretability

We have expanded our analysis to include deeper comparative studies of how GPT-2 processes Python structures:

1.  **Global vs. Local Monitoring**: While our initial setup monitored every layer to see general activation trends, we successfully transitioned to **Targeted Hooking**. This allowed us to isolate Layer 5 to see exactly how it represents specific tokens like `for` and `print`.
2.  **Logit Difference Analysis**: We introduced a quantitative way to measure model preference. By comparing the 'logits' for `print` versus `return`, we observed how the model's confidence shifts based on the preceding code context.
3.  **Causal Sensitivity**: By testing 'Clean' vs 'Corrupted' code (fixing or removing indentation), we began to see how sensitive the model's internal states are to syntactical correctness.
4.  **Visualizing Internals**: Using heatmaps of the residual stream, we moved from looking at single average numbers to seeing the 'fingerprint' of activations across 768 dimensions for every word in our code snippet.